# Combine HCRIS v1996 + v2010 and clean duplicates

In [ ]:
import pandas as pd
import numpy as np

v1996 = pd.read_csv('../data/output/HCRIS_Data_v1996.csv')
v2010 = pd.read_csv('../data/output/HCRIS_Data_v2010.csv')

# add missing columns from v2010 forms
v1996['hvbp_payment'] = np.nan
v1996['hrrp_payment'] = np.nan

# combine and parse dates
hcris = pd.concat([v1996, v2010], ignore_index=True)
for col in ['fy_end', 'fy_start', 'date_processed', 'date_created']:
    hcris[col] = pd.to_datetime(hcris[col], format='mixed', dayfirst=False)

hcris['tot_discounts'] = hcris['tot_discounts'].abs()
hcris['hrrp_payment'] = hcris['hrrp_payment'].abs()
hcris['fyear'] = hcris['fy_end'].dt.year
hcris = hcris.drop(columns=['year']).sort_values(['provider_number', 'fyear']).reset_index(drop=True)

print(hcris.groupby('fyear').size())

# Clean duplicate reports

In [ ]:
# count reports per hospital-fyear
hcris['total_reports'] = hcris.groupby(['provider_number', 'fyear'])['provider_number'].transform('count')
hcris['report_number'] = hcris.groupby(['provider_number', 'fyear']).cumcount() + 1

# --- Set 1: hospitals with only one report ---
unique1 = hcris[hcris['total_reports'] == 1].copy()
unique1 = unique1.drop(columns=['report', 'total_reports', 'report_number', 'npi', 'status'])
unique1['source'] = 'unique reports'

# --- Duplicates: hospitals with multiple reports ---
dupes = hcris[hcris['total_reports'] > 1].copy()
dupes['time_diff'] = (dupes['fy_end'] - dupes['fy_start']).dt.days
dupes['total_days'] = dupes.groupby(['provider_number', 'fyear'])['time_diff'].transform('sum')

In [ ]:
# --- Set 2: multiple reports summing to <370 days -> aggregate ---
dupes_short = dupes[dupes['total_days'] < 370].copy()
dupes_short['hrrp_payment'] = dupes_short['hrrp_payment'].fillna(0)
dupes_short['hvbp_payment'] = dupes_short['hvbp_payment'].fillna(0)

sum_cols = ['tot_charges', 'tot_discounts', 'tot_operating_exp', 'ip_charges',
            'icu_charges', 'ancillary_charges', 'tot_discharges', 'mcare_discharges',
            'mcaid_discharges', 'tot_mcare_payment', 'secondary_mcare_payment',
            'hvbp_payment', 'hrrp_payment']

agg_dict = {c: 'sum' for c in sum_cols}
agg_dict['beds'] = 'max'
agg_dict['fy_start'] = 'min'
agg_dict['fy_end'] = 'max'
agg_dict['date_processed'] = 'max'
agg_dict['date_created'] = 'min'
agg_dict['street'] = 'first'
agg_dict['city'] = 'first'
agg_dict['state'] = 'first'
agg_dict['zip'] = 'first'
agg_dict['county'] = 'first'

unique2 = dupes_short.groupby(['provider_number', 'fyear'], as_index=False).agg(agg_dict)
unique2['source'] = 'total for year'

In [ ]:
# --- Set 3+4: multiple reports with >=370 total days ---
dupes_long = dupes[dupes['total_days'] >= 370].copy()
dupes_long['max_days'] = dupes_long.groupby(['provider_number', 'fyear'])['time_diff'].transform('max')
dupes_long['max_date'] = dupes_long.groupby(['provider_number', 'fyear'])['fy_end'].transform('max')

# Set 3: one report covers the full year
mask3 = ((dupes_long['max_days'] == dupes_long['time_diff']) &
         (dupes_long['time_diff'] > 360) &
         (dupes_long['max_date'] == dupes_long['fy_end']))
unique3 = dupes_long[mask3].copy()
unique3 = unique3.drop(columns=['report', 'total_reports', 'report_number', 'npi', 'status',
                                 'max_days', 'time_diff', 'total_days', 'max_date'])
unique3['source'] = 'primary report'

# Set 4: remaining -> weighted average by days
dupes_remain = dupes_long.merge(
    unique3[['provider_number', 'fyear']].drop_duplicates(),
    on=['provider_number', 'fyear'], how='left', indicator=True
)
dupes_remain = dupes_remain[dupes_remain['_merge'] == 'left_only'].drop(columns=['_merge']).copy()

weight_cols = ['tot_charges', 'tot_discounts', 'tot_operating_exp', 'ip_charges',
               'icu_charges', 'ancillary_charges', 'tot_discharges', 'mcare_discharges',
               'mcaid_discharges', 'tot_mcare_payment', 'secondary_mcare_payment',
               'hvbp_payment', 'hrrp_payment']

for c in weight_cols:
    dupes_remain[c] = dupes_remain[c] * (dupes_remain['time_diff'] / dupes_remain['total_days'])

dupes_remain['hrrp_payment'] = dupes_remain['hrrp_payment'].fillna(0)
dupes_remain['hvbp_payment'] = dupes_remain['hvbp_payment'].fillna(0)

unique4 = dupes_remain.groupby(['provider_number', 'fyear'], as_index=False).agg(agg_dict)
unique4['source'] = 'weighted_average'

# Save final per-year CSVs

In [ ]:
final = pd.concat([unique1, unique2, unique3, unique4], ignore_index=True)
final = final.rename(columns={'fyear': 'year'}).sort_values(['provider_number', 'year']).reset_index(drop=True)

for y in sorted(final['year'].unique()):
    final[final['year'] == y].to_csv(f'../data/output/data-{y}.csv', index=False)

print('Years written:', sorted(final['year'].unique()))
print('Total rows:', len(final))